# AquaInsight — Task 9: Data Validation and Anomaly Detection

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


In [ ]:
# Build a sample-level water-quality matrix for the task
selected = [
    "pH",
    "Turbidity",
    "Specific conductance",
    "Nitrate",
    "Total Phosphorus, mixed forms",
    "Dissolved oxygen (DO)"
]

wide = (
    df[df["CharacteristicName"].isin(selected)]
    .assign(
        sample_key=lambda x:
            x["MonitoringLocationID"].astype(str) + "_" +
            x["ActivityStartDate"].astype(str)
    )
    .pivot_table(
        index="sample_key",
        columns="CharacteristicName",
        values="ResultValue",
        aggfunc="median"
    )
)

display(wide.head())
print("Wide matrix shape:", wide.shape)


# Task 9 — Data Validation and Anomaly Detection

### Internship requirement
Integrate automated anomaly detection with manual verification using **IQR, Z-score, and Mahalanobis distance**.

### Steps
1. Create a numeric sample-level feature matrix.
2. Handle missing values for the anomaly-detection demonstration.
3. Calculate IQR-based outlier flags.
4. Calculate Z-score flags.
5. Calculate multivariate Mahalanobis-distance flags.
6. Combine the methods into a confidence-style flag count.
7. Review high-confidence candidates against original records.
8. Correct or remove data only after domain/manual verification.


In [18]:
anom = wide.dropna(axis=1, how="all").copy()
anom = anom.sample(min(10000, len(anom)), random_state=42)

# Median imputation for the anomaly-detection demonstration
anom_imp = anom.fillna(anom.median(numeric_only=True))

# IQR flags
q1 = anom_imp.quantile(0.25)
q3 = anom_imp.quantile(0.75)
iqr = q3 - q1
iqr_flags = ((anom_imp < (q1 - 1.5 * iqr)) | (anom_imp > (q3 + 1.5 * iqr))).any(axis=1)

# Z-score flags
z_scores = np.abs(stats.zscore(anom_imp, nan_policy="omit"))
z_scores = np.nan_to_num(z_scores)
z_flags = (z_scores > 3).any(axis=1)

# Mahalanobis distance
X = anom_imp.to_numpy(dtype=float)
mu = X.mean(axis=0)
cov = np.cov(X, rowvar=False)
cov += np.eye(cov.shape[0]) * 1e-8
inv_cov = np.linalg.pinv(cov)
md = np.array([mahalanobis(row, mu, inv_cov) for row in X])
threshold = np.sqrt(stats.chi2.ppf(0.997, df=X.shape[1]))
md_flags = md > threshold

anomaly_report = pd.DataFrame({
    "IQR_flag": iqr_flags,
    "Zscore_flag": z_flags,
    "Mahalanobis_flag": md_flags,
    "flag_count": iqr_flags.astype(int) + z_flags.astype(int) + md_flags.astype(int),
    "Mahalanobis_distance": md
}, index=anom.index)

display(anomaly_report["flag_count"].value_counts().sort_index().to_frame("records"))
display(anomaly_report.sort_values(
    ["flag_count", "Mahalanobis_distance"], ascending=False
).head(20))

print("Recommended workflow: inspect high-confidence anomaly candidates against the original laboratory/sample records before correction or removal.")


,records
flag_count,
0,4216
1,2473
2,252
3,333


,IQR_flag,Zscore_flag,Mahalanobis_flag,flag_count,Mahalanobis_distance
sample_key,,,,,
NS01EJ0157_2007-04-27,True,True,True,3,52.548858
PE01CB0143_2016-03-10,True,True,True,3,49.188190
NS01ED0085_2008-06-05,True,True,True,3,31.069920
NS01EJ0157_2017-02-06,True,True,True,3,29.078987
NS01EJ0157_2010-11-17,True,True,True,3,23.296196
NS01ED0110_2017-07-06,True,True,True,3,19.690107
NS01DD0016_2007-10-19,True,True,True,3,19.395332
PE01CB0143_2019-11-06,True,True,True,3,17.877432
NS01EJ0001_2015-03-24,True,True,True,3,16.066851


Recommended workflow: inspect high-confidence anomaly candidates against the original laboratory/sample records before correction or removal.


# Final Assessment — Is the Internship Task Complete?

## Coverage of the 9 requested tasks

| Task | Status | Coverage |
|---|---|---|
| 1. Feature scaling | ✅ | Min-Max, Standardization, Robust Scaling |
| 2. Skewness reduction | ✅ | Box-Cox + Yeo-Johnson + skewness/kurtosis comparison |
| 3. Time-based features | ✅ | Year, month, day, hour, weekday, seasonality |
| 4. Duplicate detection | ✅ | Exact matching + Levenshtein + Jaccard + combined linkage candidates |
| 5. Missing-data sensitivity | ✅* | Simulated missingness + imputation sensitivity |
| 6. Categorical transformation | ✅ | One-hot, ordinal, frequency, target encoding |
| 7. EDA | ✅ | Missingness, distributions, frequencies, units, diagnostics |
| 8. Imputation | ✅ | KNN vs MICE/Iterative Imputation with RMSE |
| 9. Data validation | ✅ | IQR + Z-score + Mahalanobis + manual-review workflow |

### What prevents me from calling it 100% complete

The supplied dataset has **no official WQI target and no WQI calculation formula**. Therefore, a genuine "Predicting Water Quality Index" model cannot responsibly be trained yet.

The project is **internship-ready as a data-quality/preprocessing project**, but the final predictive-model stage should be added once the official WQI target/formula is provided.

### Recommended next stage

After receiving the official WQI definition:

1. Calculate or load the official WQI target.
2. Build a sample-level feature matrix.
3. Split data using a leakage-resistant time/location strategy.
4. Train baseline and ensemble regression models.
5. Compare MAE, RMSE, and R².
6. Perform residual analysis.
7. Report feature importance.
8. Save the final model and preprocessing pipeline.

**Conclusion:** This revised notebook now aligns closely with the 9 internship tasks shown in the assignment and is substantially stronger than the original `data1.ipynb`.


## Task 9 — Conclusion

The analysis above completes the requested **Data Validation and Anomaly Detection** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.